# 🎵 Apple Music Songlist Extractor

Extract **all albums + all songs** of **one or multiple Apple Music artists** into the `songlist.txt` format that the [Telegram-Leecher](https://github.com/ajithvnr2001/Telegram-Leecher) bot's `/amusic songs` mode consumes.

This notebook is **standalone**: it works without the bot, requires **no login/tokens** (public iTunes Lookup API + public Apple Music pages), and dedupes tracks that appear on multiple albums.

**Pipeline:** artist/album/playlist URL(s) → albums (oldest-first) → tracks (disc/track order) → `songlist.txt` → optional S3 push / browser download.

Behind the scenes it runs `tools/extract_songlist.py` from the main repo (always up to date). Troubleshooting + crash semantics live in [guide/SONGLIST_EXTRACTION.md](https://github.com/ajithvnr2001/Telegram-Leecher/blob/main/guide/SONGLIST_EXTRACTION.md).

## Usage

1. Paste **one Apple Music artist/album/playlist URL per line** into `ARTIST_URLS` below (any number, of any artists).
2. (Optional) Upload to S3 for zero-touch pickup by the bot (`SONGLIST_URL`) — or just click the generated file in the left Files panel to download it.
3. **Run all cells** (`Runtime → Run all`).

Output shape:
```
# N album group(s), M unique songs — generated YYYY-MM-DD
Album Name (Year):
  https://music.apple.com/in/song/<adamID>
  ...
```

Then: put `songlist.txt` at `/content/songlist.txt` on the bot runtime — or point the bot's `SONGLIST_URL` at the S3 copy you push from here.

In [ ]:
# @title 🎛️ Extract songlist (multi-artist)

# @markdown ### Input
# @markdown One Apple Music URL per line — **artist** (→ ALL its albums, oldest first), album, or playlist. Any number of artists can be mixed.
ARTIST_URLS = """https://music.apple.com/in/artist/a-r-rahman/3249567
https://music.apple.com/in/artist/ilaiyaraaja/20317131"""  # @param {type: "string"}
COUNTRY = "IN"  # @param {type: "string"}  # @markdown iTunes storefront (e.g. IN / US / GB / JP)
LIMIT_ALBUMS = 0  # @param {type: "integer"}  # @markdown cap albums per artist (0 = ALL)
MERGE_TO_ONE = True  # @param {type: "boolean"}  # @markdown ONE merged songlist.txt (True) or a separate file per artist (False)

# @markdown ---
# @markdown ### Output
OUTPUT_NAME = "songlist.txt"  # @param {type: "string"}
DOWNLOAD_LOCALLY = True  # @param {type: "boolean"}  # @markdown offer the file as a browser download after generation

# @markdown ---
# @markdown ### Optional: push result to S3 (Wasabi/B2/etc.) for the bot's SONGLIST_URL
PUSH_TO_S3 = False  # @param {type: "boolean"}
S3_ACCESS_KEY = ""  # @param {type: "string"}
S3_SECRET_KEY = ""  # @param {type: "string"}
S3_BUCKET_NAME = ""  # @param {type: "string"}
S3_ENDPOINT_URL = ""  # @param {type: "string"}
S3_REGION = "us-east-1"  # @param {type: "string"}
S3_DEST_KEY = "songlist.txt"  # @param {type: "string"}

import os, re, subprocess, sys, time

# 1) fetch the CURRENT extractor straight from the GitHub API (the raw CDN
#    can serve a stale copy for a while; api.github.com never does)
FETCH = "https://api.github.com/repos/ajithvnr2001/Telegram-Leecher/contents/tools/extract_songlist.py?ref=main"
subprocess.run(
    f"curl -fsSL -H 'Accept: application/vnd.github.raw' '{FETCH}' -o /tmp/extract_songlist.py && echo fetched",
    shell=True, check=True)
print(open("/tmp/extract_songlist.py").read(120))


# 2) normalise the multi-artist textarea into a links file
urls = [u.strip() for u in ARTIST_URLS.splitlines() if u.strip() and "music.apple.com" in u]
if not urls:
    raise SystemExit("No Apple Music URLs in ARTIST_URLS — paste at least one artist/album/playlist link.")

def _kind(u):
    m = re.search(r"music\.apple\.com/\w{2}/(artist|album|playlist)/[^/]+/([\w.-]+)", u)
    return (m.group(1), m.group(2)) if m else (None, None)

artists = [u for u in urls if _kind(u)[0] == "artist"]
others  = [u for u in urls if _kind(u)[0] != "artist"]
print(f"sources: {len(artists)} artist(s), {len(others)} album/playlist link(s)")


def _run(urls_list, out_path):
    cmd = [sys.executable, "/tmp/extract_songlist.py",
           *urls_list, "--country", COUNTRY, "-l", "=-", "-o", out_path]
    if LIMIT_ALBUMS:
        cmd += ["--limit-albums", str(LIMIT_ALBUMS)]
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout)
    if r.returncode != 0:
        print(r.stderr)
        raise RuntimeError("extract step failed — see output above")


if MERGE_TO_ONE or len(urls) == 1 or not artists:
    # everything into one songlist (extractor dedupes globally anyway)
    _run(urls, "/content/" + OUTPUT_NAME)
    outputs = ["/content/" + OUTPUT_NAME]
else:
    # one file per artist + the misc links appended to the first artist's file
    outputs = []
    for i, u in enumerate(artists, 1):
        _k, aid = _kind(u)
        out = f"/content/songlist_{i}_{aid}.txt"
        _run(([u] + others) if i == 1 else [u], out)
        outputs.append(out)

# 3) summary
total = 0
for p in outputs:
    n = sum(1 for l in open(p) if l.strip().startswith("http"))
    total += n
    print(f"{p}: {n} songs")
print(f"TOTAL: {total} unique songs across {len(outputs)} file(s)")

# 4) optional push to S3
if PUSH_TO_S3:
    import boto3
    c = boto3.client("s3", aws_access_key_id=S3_ACCESS_KEY,
                     aws_secret_access_key=S3_SECRET_KEY,
                     endpoint_url=S3_ENDPOINT_URL or None, region_name=S3_REGION)
    for p in outputs:
        dkey = S3_DEST_KEY if len(outputs) == 1 else os.path.basename(p)
        c.upload_file(p, S3_BUCKET_NAME, dkey)
        print(f"pushed {p} -> s3://{S3_BUCKET_NAME}/{dkey}")
    print(f"(Set SONGLIST_URL = \"s3://{S3_BUCKET_NAME}/{S3_DEST_KEY}\" in the bot cell.)")

# 5) optional browser download
if DOWNLOAD_LOCALLY:
    try:
        from google.colab import files as _gfiles
        for p in outputs:
            _gfiles.download(p)
    except ImportError:
        print("(browser download only available inside Google Colab UI)")


## Notes

- **Order guarantee of the bot:** download → upload to Telegram → S3 log/markers. Nothing is marked done before upload.
- **Crash-resume:** the bot tracks progress in `music-logs/songlist-dedupe.log`; a restart continues missing songs only.
- Multi-album-artist lists can be long (Rahman ≈ 200 albums ≈ 1450 songs ≈ 2–3 min to extract).
- To *re-do* a format-failed track, delete its `DONE <fmt> <adamID>` line from the dedupe log in S3.
- Tip: keep LIMIT_ALBUMS=2 for a smoke test before a full run.